# Notebook 1: Einführung und Daten

**Statistik für Data Science** · Kurs 3,906 · Bachelor Computer Science · HS2026
Universität St. Gallen

---
### Projektteam:
- Andrin Greitmann
- Noah Appenzeller
- Manuel Schmid

---

Dieses Notebook führt die Inhalte von **VL01 (Einführung & Daten)** in Python aus. 

## 0. Setup
Importiert `pandas`, `numpy`, `seaborn` und `matplotlib` und setzt den Seed.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Seed: Startwert für den Zufallsgenerator. Gleicher Seed bei jedem Durchlauf damit Resultate deterministisch & reproduzierbar sind. 
# Den Wert selbst haben wir beliebig gewählt, wir haben ihn als erstes vor der Analyse gewählt und danach nicht mehr geändert.
# Der Seed wird für alles Zufällige in allen Notebooks verwendet.
SEED = 2001
RNG = np.random.default_rng(SEED)

# Plot-Defauls
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (7.5, 4.2)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

In [3]:
import sys

print("Python  ", sys.version.split()[0])
for modul in (np, pd, sns):
    print(f"{modul.__name__:8s}", modul.__version__)
print("Seed    ", SEED)

Python   3.14.4
numpy    2.5.3
pandas   3.0.6
seaborn  0.13.2
Seed     2001


### Unseren Datensatz laden

In [4]:
from ucimlrepo import fetch_ucirepo

# Datensatz direkt aus dem UCI Repository laden (ID 117 = Census-Income KDD)
census_income_kdd = fetch_ucirepo(id=117)

# Etwas spezielles an unserem Datenset: ucimlrepo liefert die Daten aufgeteilt in Features und Target. 
# (Wahrscheinlich für Machine Learning Projekte/Algorithmen)
# Target ist das Einkommen (über/unter 50K), Features sind alle anderen Merkmale.
X = census_income_kdd.data.features
y = census_income_kdd.data.targets

# Für die EDA wollten wir alle Spalten in einem DataFrame damit das Einkommen mit den anderen Merkmalen verglichen werden kann.
# join() funktioniert hier zum Glück recht einfach, weil X und y dieselben Zeilen in derselben Reihenfolge haben.
census = X.join(y)

# Metadaten: Herkunft, Beschreibung und Erhebung des Datensatzes (Damit wir auf Bias analysieren können)
print("Metadata: " + str(census_income_kdd.metadata))

print(f"Rows: {census.shape[0]}, Columns: {census.shape[1]}")

# Variablenbeschreibung: Name, Typ und Bedeutung jeder Spalte (Grundlage um später Messniveau zu bestimmen)
print("Variable descriptions:")

display(census_income_kdd.variables)

Metadata: {'uci_id': 117, 'name': 'Census-Income (KDD)', 'repository_url': 'https://archive.ics.uci.edu/dataset/117/census+income+kdd', 'data_url': 'https://archive.ics.uci.edu/static/public/117/data.csv', 'abstract': 'This data set contains weighted census data extracted from the 1994 and 1995 current population surveys conducted by the U.S. Census Bureau.', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 299285, 'num_features': 41, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Education Level', 'Marital Status', 'Race', 'Sex', 'Other', 'Nationality', 'Income'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2000, 'last_updated': 'Fri Mar 08 2024', 'dataset_doi': '10.24432/C5N30T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set contains weighted census data extracted

,name,role,type,demographic,description,units,missing_values
0,AAGE,Feature,Integer,Age,age,None,no
1,ACLSWKR,Feature,Categorical,NaN,class of worker,None,no
2,ADTINK,Feature,Integer,NaN,industry code,None,no
3,ADTOCC,Feature,Integer,Occupation,occupation code,None,no
4,AHGA,Feature,Integer,Education Level,education,None,no
5,AHSCOL,Feature,Categorical,Education Level,enrolled in edu last week,None,no
6,AMARITL,Feature,Categorical,Marital Status,marital status,None,no
7,AMJIND,Feature,Categorical,NaN,major industry code,None,no
8,AMJOCC,Feature,Categorical,Occupation,major occupation code,None,no
9,ARACE,Feature,Categorical,Race,race,None,no


## 1. Datentypen und Messniveaus
Erster Blick auf unseren Datensatz:

In [5]:
census.info()

<class 'pandas.DataFrame'>
RangeIndex: 199523 entries, 0 to 199522
Data columns (total 42 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   AAGE       199523 non-null  int64  
 1   ACLSWKR    199523 non-null  str    
 2   ADTINK     199523 non-null  int64  
 3   ADTOCC     199523 non-null  int64  
 4   AHGA       199523 non-null  str    
 5   AHSCOL     199523 non-null  str    
 6   AMARITL    199523 non-null  str    
 7   AMJIND     199523 non-null  str    
 8   AMJOCC     199523 non-null  str    
 9   ARACE      199523 non-null  str    
 10  AREORGN    199523 non-null  str    
 11  ASEX       199523 non-null  str    
 12  AUNMEM     199523 non-null  str    
 13  AUNTYPE    199523 non-null  str    
 14  AWKSTAT    199523 non-null  str    
 15  CAPGAIN    199523 non-null  int64  
 16  GAPLOSS    199523 non-null  int64  
 17  DIVVAL     199523 non-null  int64  
 18  FILESTAT   199523 non-null  str    
 19  GRINREG    199523 non-null  str   

### Wichtigste Erkenntnisse zur Datenqualität und Vollständigkeit

1. **Nicht jede Spalte ist vollständig:**
Ein signifikanter Teil des Datensatzes weist Lücken auf. Wir haben hier drei Spalten als Beispiele herausgesucht:
    - **`MIGMTR1`** (Migration Code - Change in MSA) hat nur 99'827 von 199'523 Einträgen (es fehlen knapp 50 % der Daten).
    - **`PEFNTVTY`** (Country of Birth Father) hat 192'810 von 199'523 Einträgen (es fehlen 6.713 Werte).
    - **`GRINST`** (State of Previous Residence) hat 198'815 von 199'523 Einträgen (es fehlen 708 Werte).

2. **Systematische Lücken bei migrationsspezifischen Daten:**
Auffällig ist, dass die Spalten `MIGMTR1`, `MIGMTR3`, `MIGMTR4` und `MIGSUN` alle exakt dieselbe Anzahl an nicht-leeren Werten (99'827) aufweisen. Dies deutet stark darauf hin, dass diese Werte nicht zufällig fehlen sondern aufgrund einer gemeinsamen Abhängigkeit. 

3. **Fehlerhafte Metadaten-Dokumentation (`missing_values` Spalte):**
Ein Problem erkannten wir bei der Dokumentationsspalte `missing_values`. Dort wird bei einigen Spalten fälschlicherweise der Wert `no` angegeben, obwohl `df.info()` eindeutig Lücken belegt. Zum Beispiel die Spalte `PENATVTY`, bei der 3'393 Werte fehlen. Man darf sich bei der Vorverarbeitung also nicht blind auf die textuelle Dokumentation verlassen.

4. **Der `dtype` ist nicht das Messniveau:** Zum Beispiel ist `ADTOCC` als `int64` typisiert, aber es handelt sich um eine nominale Skala (Kategoriencodes für Berufe ohne jegliche Rangfolge). Hier würde Pandas zwar einen Mittelwert berechnen, dies wäre mathematisch aber nicht zulässig, da es keinen „durchschnittlichen Beruf“ gibt.

In [8]:
def spalten_ueberblick(df: pd.DataFrame, beispiele: int = 3) -> pd.DataFrame:
    """Kompakter Steckbrief je Spalte: dtype, Anzahl eindeutiger Werte, fehlende Werte, Beispiele.

    Erster Schritt bei jedem unbekannten Datensatz.
    """
    zeilen = []
    for spalte in df.columns:
        s = df[spalte]
        werte = s.dropna().unique()[:beispiele]
        zeilen.append(
            {
                "dtype": str(s.dtype),
                "eindeutig": s.nunique(dropna=True),
                "fehlend": int(s.isna().sum()),
                "fehlend_%": round(s.isna().mean() * 100, 1),
                "beispiele": ", ".join(map(str, werte)),
            }
        )
    return pd.DataFrame(zeilen, index=df.columns)


spalten_ueberblick(census)

,dtype,eindeutig,fehlend,fehlend_%,beispiele
AAGE,int64,91,0,0.0,"73, 58, 18"
ACLSWKR,str,9,0,0.0,"Not in universe, Self-employed-not incorpora..."
ADTINK,int64,52,0,0.0,"0, 4, 40"
ADTOCC,int64,47,0,0.0,"0, 34, 10"
AHGA,str,17,0,0.0,"High school graduate, Some college but no de..."
AHSCOL,str,3,0,0.0,"Not in universe, High school, College or un..."
AMARITL,str,7,0,0.0,"Widowed, Divorced, Never married"
AMJIND,str,24,0,0.0,"Not in universe or children, Construction, ..."
AMJOCC,str,15,0,0.0,"Not in universe, Precision production craft ..."
ARACE,str,5,0,0.0,"White, Asian or Pacific Islander, Amer Indi..."


### Die vier Messniveaus

| Skala | Eigenschaft | Erlaubte Statistik |
|---|---|---|
| **Nominal** | Kategorien ohne Ordnung | Modus, Häufigkeiten |
| **Ordinal** | Rangfolge, Abstände undefiniert | zusätzlich Median, Perzentile |
| **Intervall** | gleiche Abstände, kein natürlicher Nullpunkt | zusätzlich Mittelwert, Standardabweichung |
| **Ratio** | absoluter Nullpunkt | zusätzlich Verhältnisse, geometrisches Mittel |

In [10]:
# Inhaltliche Zuordnung. Sie steht bewusst als Hand-Mapping da, weil sie sich
# nicht aus dem dtype ableiten laesst.
MESSNIVEAU = {
    "AAGE": "ratio",          # 0 Jahre ist ein echter Nullpunkt
    "ACLSWKR": "nominal",     # Arbeiterklasse (z.B. Private, Federal Gov)
    "ADTINK": "nominal",      # Industrie-Code, int64 aber reiner Code ohne Rang
    "ADTOCC": "nominal",      # Berufs-Code, int64 aber reiner Code ohne Rang
    "AHGA": "ordinal",        # Bildungsabschluss (hat eine logische Rangfolge)
    "AHSCOL": "nominal",      # Einschreibung Schule/Uni
    "AMARITL": "nominal",     # Familienstand
    "AMJIND": "nominal",      # Hauptbranche
    "AMJOCC": "nominal",      # Hauptberufsgruppe
    "ARACE": "nominal",       # Ethnie/Rasse
    "AREORGN": "nominal",     # Hispanische Herkunft
    "ASEX": "nominal",        # Geschlecht
    "AUNMEM": "nominal",      # Gewerkschaftsmitgliedschaft
    "AUNTYPE": "nominal",     # Grund für Arbeitslosigkeit
    "AWKSTAT": "nominal",     # Vollzeit-/Teilzeit-Status
    "CAPGAIN": "ratio",       # Kapitalgewinne (0 $ ist ein echter Nullpunkt)
    "GAPLOSS": "ratio",       # Kapitalverluste (0 $ ist ein echter Nullpunkt)
    "DIVVAL": "ratio",        # Dividenden aus Aktien (0 $ ist ein echter Nullpunkt)
    "FILESTAT": "nominal",    # Steuerstatus
    "GRINREG": "nominal",     # Region des vorherigen Wohnsitzes
    "GRINST": "nominal",      # Bundesstaat des vorherigen Wohnsitzes
    "HHDFMX": "nominal",      # Detaillierte Haushaltsstruktur
    "HHDREL": "nominal",      # Beziehung im Haushalt
    "MARSUPWRT": "ratio",     # Gewichtung / Instance Weight
    "MIGMTR1": "nominal",     # Migrationscode (MSA)
    "MIGMTR3": "nominal",     # Migrationscode (Region)
    "MIGMTR4": "nominal",     # Migrationscode (innerhalb Region)
    "MIGSAME": "nominal",     # Gleiches Haus wie vor 1 Jahr (Ja/Nein-Code)
    "MIGSUN": "nominal",      # Migration (Sunbelt)
    "NOEMP": "ratio",         # Anzahl der Beschäftigten beim Arbeitgeber
    "PARENT": "nominal",      # Elternpräsenz im Haushalt
    "PEFNTVTY": "nominal",    # Geburtsland Vater
    "PEMNTVTY": "nominal",    # Geburtsland Mutter
    "PENATVTY": "nominal",    # Geburtsland Selbst
    "PRCITSHP": "nominal",    # Staatsbürgerschaft
    "SEOTR": "nominal",       # Selbstständigkeit (Code)
    "VETQVA": "nominal",      # Fragebogen für Veteranen
    "VETYN": "nominal",       # Veteranenstatus (Ja/Nein)
    "WKSWORK": "ratio",       # Gearbeitete Wochen im Jahr (0 ist echter Nullpunkt)
    "AHRSPAY": "ratio",       # Stundenlohn (0 $ ist echter Nullpunkt)
    "year": "intervall",      # Jahreszahlen haben keinen natürlichen Nullpunkt
    "income": "ordinal",      # Einkommensklasse (z.B. <50k, >=50k), hat Rangfolge
}

zuordnung = pd.DataFrame(
    {
        "dtype": census.dtypes.astype(str),
        "messniveau": pd.Series(MESSNIVEAU),
    }
)
zuordnung["mittelwert_zulaessig"] = zuordnung["messniveau"].isin(["intervall", "ratio"])
zuordnung

,dtype,messniveau,mittelwert_zulaessig
AAGE,int64,ratio,True
ACLSWKR,str,nominal,False
ADTINK,int64,nominal,False
ADTOCC,int64,nominal,False
AHGA,str,ordinal,False
AHSCOL,str,nominal,False
AMARITL,str,nominal,False
AMJIND,str,nominal,False
AMJOCC,str,nominal,False
ARACE,str,nominal,False


Wie unsere Übersicht zeigt, führt der Datentyp `int64` dazu, dass `pandas` mathematisch völlig unzulässige Mittelwerte für rein nominale Variablen (wie z.B. den Berufscode `ADTOCC`) berechnet. Es gibt jedoch auch Spalten wie z.B. `CAPGAIN` (Kapitalgewinne), die den `int64` Datentyp haben, bei denen ein Mittelwert durchaus sinnvoll ist. 